# 01 — Data owner: encrypt and decrypt

## Goal

Create one trusted `HESession`, encrypt private values, and persist only public HE material plus ciphertext. Keep this kernel running while the compute notebook executes.

## Setup

From the repository root, prepare the current SDK checkout before starting JupyterLab:

```sh
python3 -m venv .venv-he-notebook
. .venv-he-notebook/bin/activate
python -m pip install --upgrade pip
python -m pip install -e '.[openfhe]' 'jupyterlab>=4,<5'
python -m jupyter lab
```

After version 0.4.0 is published, the editable install can be replaced with `python -m pip install 'he_looming_sdk[openfhe]==0.4.0'`.

## Steps

### 1. Choose private input and a shared filesystem workspace

In [ ]:
import json
import math
import os
from pathlib import Path

from he_sdk import HESession
from he_sdk.artifacts import list_ciphertexts

WORKSPACE = Path(
    os.getenv("HE_SDK_WORKSPACE", Path.home() / "he-sdk-notebook-workspace")
).expanduser().resolve()
PRIVATE_VALUES = [10.0, 20.0, 30.0]
EXPECTED = {
    "sum": sum(PRIVATE_VALUES),
    "mean": sum(PRIVATE_VALUES) / len(PRIVATE_VALUES),
    "variance": sum(
        (value - sum(PRIVATE_VALUES) / len(PRIVATE_VALUES)) ** 2
        for value in PRIVATE_VALUES
    ) / len(PRIVATE_VALUES),
}

print("workspace:", WORKSPACE)
print("private values are visible only in this owner notebook")

### 2. Create the owner session, encrypt, and save

Do not close this kernel yet. Version 0.4.0 deliberately does not persist the secret key.

In [ ]:
if "owner" in globals():
    owner.close()

owner = HESession.create(backend="openfhe")
encrypted_input = owner.encrypt(PRIVATE_VALUES)
owner.save(encrypted_input, WORKSPACE, name="input")

del PRIVATE_VALUES
print("saved artifacts:", list_ciphertexts(WORKSPACE))
print("owner session remains open for final decryption")

## Checks

Inspect the SDK manifest. It may reveal metadata such as vector length, backend, and scheme, but it must declare that it contains neither plaintext nor a secret key.

In [ ]:
manifest = json.loads((WORKSPACE / "manifest.json").read_text())
assert manifest["contains_plaintext"] is False
assert manifest["contains_secret_key"] is False
assert list_ciphertexts(WORKSPACE) == ("input",)

print(json.dumps({
    "format_version": manifest["format_version"],
    "backend": manifest["backend"],
    "contains_plaintext": manifest["contains_plaintext"],
    "contains_secret_key": manifest["contains_secret_key"],
    "ciphertexts": list_ciphertexts(WORKSPACE),
}, indent=2))
print("OWNER_ENCRYPT=PASS")

### 3. Pause here

Leave this kernel running. Open `02_compute_encrypted.ipynb` with a **different kernel**, run it top-to-bottom, and then return here.

### 4. Load and decrypt the encrypted results

Run this only after the compute notebook prints `COMPUTE_ONLY=PASS`.

In [ ]:
available = set(list_ciphertexts(WORKSPACE))
required = {"sum", "mean", "variance"}
assert required <= available, f"run compute notebook first; missing {sorted(required - available)}"

observed = {
    operation: owner.decrypt(owner.load(WORKSPACE, name=operation))
    for operation in sorted(required)
}
for operation, expected in EXPECTED.items():
    assert math.isclose(observed[operation], expected, rel_tol=1e-3, abs_tol=1e-3)

print(json.dumps(observed, indent=2, sort_keys=True))
print("OWNER_DECRYPT=PASS")

## Next steps

Close `owner` when finished. If the owner kernel is lost before decryption, version 0.4.0 intentionally provides no recovery because the secret key was never written to the shared workspace. Tomorrow this same workspace path can be mounted from a PVC without changing notebook SDK calls.

In [ ]:
owner.close()
print("owner session closed")